# DS 4420 — PCOS Classification: 1D CNN (Keras)
**Dataset:** [Kaggle PCOS Dataset](https://www.kaggle.com/datasets/shreyasvedpathak/pcos-dataset)  
**Model:** 1D Convolutional Neural Network using Keras  
**Task:** Binary classification — PCOS (Y/N)

Since our dataset is tabular (not image-based), we use a **1D CNN**.  
The idea is to reshape each patient's feature vector into a sequence, then let
convolutional filters slide over groups of neighboring features to pick up local patterns.
This is different from an MLP, which treats every feature independently.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## 1. Load and Clean the Data

In [ ]:
# load the dataset
# if running locally, update the path below
# if running on Colab, upload the CSV first and use: df = pd.read_csv('PCOS_data.csv')
df = pd.read_csv('PCOS_data.csv')

print('Shape:', df.shape)
df.head()

In [ ]:
# drop columns that are just IDs or almost entirely empty
df = df.drop(columns=['Sl. No', 'Patient File No.', 'Unnamed: 44'], errors='ignore')

# separate the target
y = df['PCOS (Y/N)'].values
X = df.drop(columns=['PCOS (Y/N)'])

# force everything to numeric (some columns have stray characters)
X = X.apply(pd.to_numeric, errors='coerce')

# fill any remaining NaNs with the column median
X = X.fillna(X.median())

print('Features shape:', X.shape)
print('Class balance  — PCOS=1:', y.sum(), '| No PCOS=0:', (y == 0).sum())

## 2. Train/Test Split and Normalization

In [ ]:
# 80/20 split, stratified so both splits have the same class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.20, random_state=42, stratify=y
)

# z-score normalization — fit only on training data to avoid data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Train:', X_train.shape, '| Test:', X_test.shape)

## 3. Reshape for 1D CNN

Keras expects 1D CNN input to have shape `(samples, steps, channels)`.  
We treat each feature as one time-step with a single channel, so we just
add a trailing dimension: `(n_samples, n_features, 1)`.

In [ ]:
# add the channel dimension
X_train_cnn = X_train[..., np.newaxis]   # (432, 43, 1)
X_test_cnn  = X_test[..., np.newaxis]    # (109, 43, 1)

print('CNN input shape (train):', X_train_cnn.shape)
print('CNN input shape (test) :', X_test_cnn.shape)

## 4. Build the 1D CNN

Architecture:
- **Conv1D (32 filters, kernel=3)** — scans groups of 3 neighboring features
- **MaxPooling1D** — keeps only the strongest activations, reduces size
- **Conv1D (64 filters, kernel=3)** — deeper feature extraction
- **GlobalAveragePooling1D** — collapses the sequence into a single vector
- **Dense (64, ReLU)** — fully connected layer
- **Dropout (0.3)** — randomly drops neurons during training to prevent overfitting
- **Dense (1, Sigmoid)** — outputs a probability between 0 and 1

In [ ]:
# get the number of features from the training data
n_features = X_train_cnn.shape[1]

# build the model
model = keras.Sequential([
    # first convolutional block
    layers.Conv1D(filters=32, kernel_size=3, activation='relu',
                  input_shape=(n_features, 1)),
    layers.MaxPooling1D(pool_size=2),

    # second convolutional block
    layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
    layers.GlobalAveragePooling1D(),

    # fully connected head
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')   # binary output
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Train the Model

In [ ]:
# early stopping will stop training if validation loss stops improving
# this helps prevent overfitting without having to guess the right number of epochs
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,           # stop after 10 epochs with no improvement
    restore_best_weights=True
)

history = model.fit(
    X_train_cnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.15,    # hold out 15% of training data for validation
    callbacks=[early_stop],
    verbose=1
)

## 6. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# loss curve
axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend()

# accuracy curve
axes[1].plot(history.history['accuracy'],     label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Evaluate on the Test Set

In [ ]:
# get predicted probabilities, then threshold at 0.5
y_prob = model.predict(X_test_cnn).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# print the full classification report
print(classification_report(y_test, y_pred, target_names=['No PCOS', 'PCOS']))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No PCOS', 'PCOS'])
disp.plot(cmap='Blues')
plt.title('CNN Confusion Matrix — Test Set')
plt.show()

## 8. What the Filters Are Looking At

One thing we can do with a 1D CNN is look at the activation of each filter
across the feature sequence. This gives a rough sense of which parts of the
feature vector the network is responding to most strongly.

In [ ]:
# build a sub-model that outputs the first Conv1D layer's activations
activation_model = keras.Model(
    inputs=model.input,
    outputs=model.layers[0].output    # first Conv1D layer
)

# get activations for the first test sample
sample = X_test_cnn[[0]]              # shape (1, n_features, 1)
activations = activation_model.predict(sample)[0]  # shape (n_features-2, 32 filters)

# plot the mean activation across all 32 filters at each position
mean_activation = activations.mean(axis=1)

plt.figure(figsize=(10, 3))
plt.plot(mean_activation)
plt.title('Mean Filter Activation Across Feature Positions (First Conv Layer)')
plt.xlabel('Feature Position')
plt.ylabel('Mean Activation')
plt.tight_layout()
plt.show()

# the feature names (minus dropped columns) for reference
feature_names = X.columns.tolist()
print('\nFeature names (in order):')
for i, name in enumerate(feature_names):
    print(f'  {i:>2}: {name}')